# Fleet maintenance cost: v36-v37

**v36: interaction terms in the smooth budget model; v37: same at risk x7**

Public leaderboard: v36 17,000.26

This notebook contains all of this version's code. Each script is written to disk with `%%writefile` (so scripts can import each other) and then run with `%run`.

**Before running:** put the competition files `train.csv`, `test.csv` and `sample_submission.csv` in a `data/` folder next to this notebook, and install `numpy pandas scipy scikit-learn lightgbm xgboost catboost`.

In [ ]:
import os
os.makedirs("oof", exist_ok=True)   # saved model predictions go here

## Shared modules

### `fleet_prep.py`
Loads the data and undoes the test-depot sensor shifts.

In [ ]:
%%writefile fleet_prep.py
"""Data loading and depot-shift corrections for the Fleet Maintenance competition.

Findings from adversarial validation (train vs test AUC was 1.000 before fixes):
  * s054..s063 look like a 10-bin histogram. In test the bins are shifted one
    column to the right (test s054 is always 0 and test s055..s063 match train
    s054..s062). We shift them back and mark test s063 as missing.
  * s120 has a constant offset of ~16,025,000 in test.
  * s122 is scaled by ~2.54 in test (looks like a unit change).
  * s047 is entirely missing in test, so it is dropped.
"""
import numpy as np
import pandas as pd

HIST = [f"s{i:03d}" for i in range(54, 64)]
S120_OFFSET = 16_025_000.0
S122_SCALE = 2.54


def fix_test(te: pd.DataFrame) -> pd.DataFrame:
    te = te.copy()
    vals = {f"s{i:03d}": te[f"s{i+1:03d}"].to_numpy() for i in range(54, 63)}
    for c, v in vals.items():
        te[c] = v
    te["s063"] = np.nan
    te["s120"] = np.maximum(te["s120"] - S120_OFFSET, 0)
    te["s122"] = te["s122"] / S122_SCALE
    return te


ROUNDING = {"s055": 10, "s062": 10, "s057": 100}  # test stores these coarser


def fix_train(tr: pd.DataFrame) -> pd.DataFrame:
    tr = tr.copy()
    for c, g in ROUNDING.items():
        tr[c] = np.round(tr[c] / g) * g
    return tr


def add_features(df: pd.DataFrame, sensor_cols) -> pd.DataFrame:
    out = df[sensor_cols].copy()
    out["n_missing"] = df[sensor_cols].isna().sum(1)
    out["n_zero"] = (df[sensor_cols] == 0).sum(1)
    h = df[HIST[:-1]]  # s063 is unavailable in test
    tot = h.sum(1, min_count=1)
    out["hist_total"] = tot
    for c in HIST[:-1]:
        out[f"{c}_frac"] = df[c] / tot.replace(0, np.nan)
    w = np.arange(len(HIST) - 1)
    out["hist_mean_bin"] = (h.fillna(0).to_numpy() * w).sum(1) / tot.replace(0, np.nan)
    return out


def load(data_dir="data"):
    tr = fix_train(pd.read_csv(f"{data_dir}/train.csv"))
    te = fix_test(pd.read_csv(f"{data_dir}/test.csv"))
    sensors = [c for c in te.columns if c.startswith("s") and c not in ("s047", "s063")]
    X = add_features(tr, sensors)
    Xt = add_features(te, sensors)
    return tr, te, X, Xt

### `policy.py`
Turns failure probability + repair-cost distribution into the cost-optimal inspect/budget decision.

In [ ]:
%%writefile policy.py
"""Cost-optimal inspect/budget decisions from probabilities and cost predictions.

Cost per vehicle (competition statement):
  not inspected: C (actual repair cost, 0 if healthy)
  inspected:     10 + 0.20 * max(C - b, 0) + 0.02 * max(b - C, 0)

For a vehicle with failure probability p and conditional repair-cost distribution
C | fail ~ exp(mu + eps) (eps = empirical out-of-fold residuals), the expected
inspection cost is minimised by the 0.20/0.22 = 0.909 quantile of the mixture
(0 with prob 1-p, C|fail with prob p). We inspect only when that expected cost is
below the expected cost of not inspecting, p * E[C | fail].
"""
import numpy as np

UNDER, OVER, FIXED = 0.20, 0.02, 10.0
TAU = UNDER / (UNDER + OVER)


def competition_cost(c, flag, budget):
    return np.where(flag == 1,
                    FIXED + UNDER * np.maximum(c - budget, 0) + OVER * np.maximum(budget - c, 0),
                    c)


def decide(p, mu, eps, alpha=1.0, eps_scale=1.0, margin=0.0, budget_mult=1.0, sigma=None):
    """Return flag, budget arrays.

    alpha       multiplies probabilities (calibration / prior adjustment)
    eps_scale   widens or narrows the residual distribution
    margin      extra cost an inspection must save before we flag
    budget_mult scales the optimal budget (kept at 1 unless CV says otherwise)
    sigma       optional per-vehicle residual scale (heteroscedastic cost model);
                then eps should be standardised residuals z = residual / sigma
    """
    sig = np.ones(len(p)) if sigma is None else np.asarray(sigma)
    p = np.clip(alpha * p, 1e-9, 1 - 1e-9)
    e = np.sort(eps * eps_scale)
    # samples of C|fail per vehicle: shape (n, k)
    out_flag = np.zeros(len(p), dtype=int)
    out_b = np.zeros(len(p))
    for s in range(0, len(p), 4000):
        sl = slice(s, s + 4000)
        C = np.exp(mu[sl, None] + sig[sl, None] * e[None, :])
        k = C.shape[1]
        ps = p[sl]
        u = 1 - (1 - TAU) / ps  # required quantile of C|fail
        idx = np.clip(np.ceil(u * k).astype(int) - 1, 0, k - 1)
        b = np.where(u > 0, C[np.arange(C.shape[0]), idx], 0.0) * budget_mult
        loss_fail = (UNDER * np.maximum(C - b[:, None], 0) + OVER * np.maximum(b[:, None] - C, 0)).mean(1)
        insp = FIXED + ps * loss_fail + (1 - ps) * OVER * b
        no_insp = ps * C.mean(1)
        f = (no_insp - insp > margin).astype(int)
        out_flag[sl] = f
        out_b[sl] = np.where(f == 1, b, 0.0)
    return out_flag, out_b

### `cv_eval.py`
Labels, repair costs and censoring weights shared by all scripts.

In [ ]:
%%writefile cv_eval.py
"""Shared helpers for comparing model versions on out-of-fold (OOF) predictions.

Each version is judged two ways:
  * OOF competition cost on the training depot (policy alpha tuned on OOF)
  * AP / log-loss of the failure probabilities
For test, every version is anchored to the same number of inspections as the
best leaderboard-validated setting (v10: 1754), so a new version changes *which*
vehicles are inspected, not *how many*.
"""
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from policy import decide, competition_cost

TARGET_INSPECTIONS = 1754

_tr = pd.read_csv("data/train.csv", usecols=["failure", "repair_cost", "followup_days"])
y = _tr.failure.to_numpy()
cost = _tr.repair_cost.fillna(0).to_numpy()
weights = np.where((y == 0) & (_tr.followup_days < 365), _tr.followup_days / 365.0, 1.0)


def oof_cost(p, mu, eps, alphas=(1.0, 1.25, 1.5, 2.0), margins=(0, 1, 2)):
    best = min((competition_cost(cost, *decide(p, mu, eps, a, 1.0, m)).mean(), a, m)
               for a in alphas for m in margins)
    return best


def report(name, p, mu, eps):
    c, a, m = oof_cost(p, mu, eps)
    print(f"{name:28s} AUC {roc_auc_score(y, p):.4f}  AP {average_precision_score(y, p):.4f}  "
          f"LL {log_loss(y, np.clip(p, 1e-7, 1 - 1e-7)):.5f}  OOF cost {c:.4f} (alpha {a}, margin {m})",
          flush=True)
    return c


def test_submission(pt, mut, eps, path, budget_mult=1.0, target=TARGET_INSPECTIONS):
    """Pick alpha so the test inspection count matches the leaderboard-validated target."""
    lo, hi = 0.1, 200.0
    for _ in range(40):
        mid = np.sqrt(lo * hi)
        n = decide(pt, mut, eps, mid, 1.0, 0.0, budget_mult)[0].sum()
        lo, hi = (mid, hi) if n < target else (lo, mid)
    f, b = decide(pt, mut, eps, hi, 1.0, 0.0, budget_mult)
    te = pd.read_csv("data/test.csv", usecols=["id"])
    sub = pd.DataFrame({"id": te.id, "flag": f.astype(int), "budget": np.round(b, 2)})
    assert len(sub) == 20000 and sub.flag.isin([0, 1]).all() and (sub.budget >= 0).all()
    sub.to_csv(path, index=False)
    print(f"wrote {path}: alpha {hi:.2f}, inspections {int(f.sum())}")
    return sub

### `make_v12_v14.py`
v12-v14 builder (also imported by make_grid.py).

In [ ]:
%%writefile make_v12_v14.py
"""v12-v14: label-free prior-shift correction + better budgets + more bagging.

v12  EM prior-shift (Saerens et al., 2002): estimate the test depot's failure rate
     from unlabelled test predictions and adjust probabilities in odds space.
     Policy settings (alpha 1.5) are the ones tuned on training OOF cost; no
     leaderboard tuning. EM says the test failure rate is ~4.4x training.
v13  v12 inspections, budgets from a heteroscedastic cost model (per-vehicle
     spread of log repair cost, sigma_model.py). Same flags, -0.8..-2.1% budget
     cost on inspected vehicles in CV.
v14  v13 with test probabilities averaged over the 15 CV fold models per
     algorithm plus the 6 full-data models (more bagging, lower variance).
"""
import numpy as np, pandas as pd
from policy import decide
from cv_eval import y, cost, weights

ALPHA, MARGIN = 1.5, 0.0
te = pd.read_csv("data/test.csv", usecols=["id"])
mu = np.load("oof/mu_train.npy"); eps = np.log(cost[y == 1]) - mu[y == 1]
mut = np.load("oof/full_mut.npy")
sig_t = np.load("oof/sigma_test.npy"); z = np.load("oof/z.npy")
ps = np.average(y, weights=weights)


def em_adjust(pt, it=500):
    pi = ps
    for _ in range(it):
        r = (pi / ps) * pt
        q = r / (r + ((1 - pi) / (1 - ps)) * (1 - pt))
        pi = q.mean()
    return q, pi


def write(name, flags, budgets):
    sub = pd.DataFrame({"id": te.id, "flag": flags.astype(int), "budget": np.round(budgets, 2)})
    assert len(sub) == 20000 and sub.flag.isin([0, 1]).all() and (sub.budget >= 0).all()
    assert (sub.loc[sub.flag == 0, "budget"] == 0).all()
    sub.to_csv(f"submission_{name}.csv", index=False)
    print(f"{name}: inspections {int(flags.sum())}, mean budget {budgets[flags == 1].mean():.0f}")


def hetero_budgets(q, flags):
    # force a budget for every vehicle, then keep it only where flagged
    _, b = decide(q, mut, z, ALPHA, 1.0, -1e9, sigma=sig_t)
    return np.where(flags == 1, b, 0.0)


# v12
q12, pi = em_adjust(np.load("oof/full_pt.npy"))
print(f"EM test failure rate {pi:.4f} ({pi / ps:.1f}x train)")
f12, b12 = decide(q12, mut, eps, ALPHA, 1.0, MARGIN)
write("v12_em_prior", f12, b12)

# v13
write("v13_em_hetero_budget", f12, hetero_budgets(q12, f12))

# v14
cv_pt = np.mean([np.load(f"oof/pt_{n}_w.npy") for n in ("lgb", "xgb", "cat")], 0)
pt14 = (15 * cv_pt + 2 * np.load("oof/full_pt.npy")) / 17   # 45 CV models + 6 full-data models
q14, pi14 = em_adjust(pt14)
print(f"v14 EM test failure rate {pi14:.4f}")
f14, _ = decide(q14, mut, eps, ALPHA, 1.0, MARGIN)
write("v14_em_hetero_bagged", f14, hetero_budgets(q14, f14))

### `cost_linear_shapes.py`
Bounded shape features and tree cost predictions, imported by budget_blend.py.

In [ ]:
%%writefile cost_linear_shapes.py
"""Linear repair-cost model on bounded features only (histogram shares, mean bin, spread,
plus log operating total). Bounded inputs cannot extrapolate into the test depot's heavy
tail the way the earlier ridge on raw counters did. Blended with the v30 tree cost model."""
import numpy as np, pandas as pd
from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, SplineTransformer
from hist_features import with_shapes
from cv_eval import y, cost
from policy import decide, competition_cost

_, _, X, Xt = with_shapes()
cols = [c for c in X.columns if c[:2] in ("A_", "B_", "C_", "D_", "E_") or c.endswith("_frac") or c == "hist_mean_bin"]


def feats(D):
    F = D[cols].copy(); F["log_total"] = np.log1p(D["hist_total"]); return F


F, Ft = feats(X), feats(Xt)
pos = np.where(y == 1)[0]; lc = np.log(cost[pos])
tree = np.load("oof/mu_train_shapes.npy"); tree_t = np.load("oof/full_mut_shapes.npy")
P = (np.load("oof/lgb2_shapes.npy") + np.load("oof/cat2_shapes.npy")) / 2
print("bounded features:", F.shape[1], "| log_total train max %.2f test max %.2f" % (F.log_total.max(), Ft.log_total.max()))


def make(kind):
    steps = [SimpleImputer(strategy="median"), StandardScaler()]
    if kind == "spline": steps.append(SplineTransformer(n_knots=4, degree=2))
    return make_pipeline(*steps, RidgeCV(np.logspace(-1, 4, 25)))


for kind in ["linear", "spline"]:
    oof = np.zeros(len(pos)); trn = np.zeros(len(X)); tst = np.zeros(len(Xt))
    for seed in [0, 1, 2]:
        for a, b in KFold(5, shuffle=True, random_state=seed).split(pos):
            m = make(kind).fit(F.iloc[pos[a]], lc[a])
            oof[b] += m.predict(F.iloc[pos[b]]) / 3; trn += m.predict(F) / 15; tst += m.predict(Ft) / 15
    trn[pos] = oof
    print(f"{kind}: alone RMSE {np.sqrt(np.mean((oof - lc) ** 2)):.4f}")
    for w in [0.2, 0.35, 0.5]:
        mu = (1 - w) * tree + w * trn; e = lc - mu[pos]
        c = min(competition_cost(cost, *decide(P, mu, e, al, 1.0, 0.0)).sum() for al in [1, 1.25, 1.5])
        hi = P > 0.1
        d_tr = np.abs(w * (trn - tree))[hi].mean(); d_te = np.abs(w * (tst - tree_t))[np.load("oof/full_pt_shapes.npy") > 0.1].mean()
        print(f"   blend w{w}: RMSE {np.sqrt(np.mean(e ** 2)):.4f} OOF cost {c:7.0f} | shift check |d| train {d_tr:.3f} test {d_te:.3f}")
        np.save(f"oof/mulin_{kind}_w{w}_train.npy", mu); np.save(f"oof/mulin_{kind}_w{w}_test.npy", (1 - w) * tree_t + w * tst)
print("v30 reference OOF cost 44517")

### `smooth_cost2.py`
Smooth (spline ridge) cost models, imported by smooth_cost3.py.

In [ ]:
%%writefile smooth_cost2.py
"""Sharper smooth repair-cost models for budgets (v34 baseline: spline knots 3 degree 1 on
bounded shape features, alone). All inputs stay bounded so nothing can extrapolate."""
import numpy as np, pandas as pd
from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, SplineTransformer, PolynomialFeatures
from sklearn.kernel_ridge import KernelRidge
from cost_linear_shapes import F, Ft
from cv_eval import y, cost
from policy import decide, competition_cost

pos = np.where(y == 1)[0]; lc = np.log(cost[pos]); posb = y == 1
tree = np.load("oof/mu_train_shapes.npy")
PT = (np.load("oof/lgb2_shapes.npy") + np.load("oof/cat2_shapes.npy")) / 2
top = None


def score(mu_b):
    best = None
    for al in [1, 1.25, 1.5]:
        f, _ = decide(PT, tree, lc - tree[pos], al, 1.0, 0.0)
        _, b = decide(PT, mu_b, lc - mu_b[pos], al, 1.0, -1e9)
        c = competition_cost(cost, f, np.where(f == 1, b, 0)).sum()
        best = c if best is None or c < best else best
    return best


def cv(name, make, D=F, Dt=Ft):
    oof = np.zeros(len(pos)); trn = np.zeros(len(D)); tst = np.zeros(len(Dt))
    for seed in [0, 1, 2]:
        for a, b in KFold(5, shuffle=True, random_state=seed).split(pos):
            m = make().fit(D.iloc[pos[a]], lc[a])
            oof[b] += m.predict(D.iloc[pos[b]]) / 3; trn += m.predict(D) / 15; tst += m.predict(Dt) / 15
    trn[pos] = oof
    print(f"{name:34s} RMSE {np.sqrt(np.mean((oof - lc) ** 2)):.4f}  OOF cost {score(trn):.0f}", flush=True)
    np.save("oof/sc2_" + "".join(ch if ch.isalnum() else "_" for ch in name) + "_train.npy", trn)
    np.save("oof/sc2_" + "".join(ch if ch.isalnum() else "_" for ch in name) + "_test.npy", tst)
    return trn, tst


pre = lambda: [SimpleImputer(strategy="median"), StandardScaler()]
cv("v34 spline k3 d1", lambda: make_pipeline(*pre(), SplineTransformer(n_knots=3, degree=1), RidgeCV(np.logspace(-1, 4, 25))))
cv("spline k2 d1 (linear)", lambda: make_pipeline(*pre(), SplineTransformer(n_knots=2, degree=1), RidgeCV(np.logspace(-1, 4, 25))))
cv("spline k4 d1", lambda: make_pipeline(*pre(), SplineTransformer(n_knots=4, degree=1), RidgeCV(np.logspace(-1, 4, 25))))
# pick top-15 bounded features by abs corr with log cost, add pairwise interactions
corr = pd.Series({c: abs(np.corrcoef(F.iloc[pos][c].fillna(F[c].median()), lc)[0, 1]) for c in F.columns}).sort_values(ascending=False)
top = list(corr.index[:15]); print("top features:", top[:8])
from sklearn.compose import ColumnTransformer
inter = lambda: make_pipeline(*pre(), ColumnTransformer([("spl", SplineTransformer(n_knots=3, degree=1), list(range(F.shape[1]))),
                                                        ("int", PolynomialFeatures(2, interaction_only=True, include_bias=False), [F.columns.get_loc(c) for c in top])]),
                              RidgeCV(np.logspace(-1, 4, 25)))
cv("spline k3 d1 + top15 interactions", inter)
for g in [0.003, 0.01]:
    cv(f"kernel ridge rbf g{g}", lambda g=g: make_pipeline(*pre(), KernelRidge(alpha=1.0, kernel="rbf", gamma=g)))

## Pipeline (run in order)

### Step 1: `oof_models.py`
Out-of-fold failure probabilities and conditional repair-cost predictions.

In [ ]:
%%writefile oof_models.py
"""Out-of-fold failure probabilities and conditional repair-cost predictions.

Writes oof/*.npy so the decision policy can be tuned without retraining.
"""
import os, sys, time
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
from fleet_prep import load

SEEDS = [0, 1, 2]
NF = 5
os.makedirs("oof", exist_ok=True)
tr, te, X, Xt = load()
y = tr.failure.to_numpy()
cost = tr.repair_cost.fillna(0).to_numpy()
# Censoring: healthy vehicles observed < 365 days are only partly informative.
w = np.where((y == 0) & (tr.followup_days < 365), tr.followup_days / 365.0, 1.0)


def clf_models(seed):
    return {
        "lgb": lgb.LGBMClassifier(n_estimators=600, learning_rate=0.02, num_leaves=15,
                                  min_child_samples=40, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=0.5, reg_lambda=5, random_state=seed,
                                  verbosity=-1),
        "xgb": xgb.XGBClassifier(n_estimators=600, learning_rate=0.02, max_depth=4,
                                 min_child_weight=3, subsample=0.8, colsample_bytree=0.5,
                                 reg_lambda=5, tree_method="hist", random_state=seed,
                                 eval_metric="logloss", n_jobs=-1),
        "cat": CatBoostClassifier(iterations=1000, learning_rate=0.03, depth=5,
                                  l2_leaf_reg=8, random_seed=seed, verbose=False,
                                  allow_writing_files=False, thread_count=-1),
    }


def run_clf(use_w):
    tag = "w" if use_w else "nw"
    oof = {k: np.zeros(len(X)) for k in ["lgb", "xgb", "cat"]}
    tst = {k: np.zeros(len(Xt)) for k in oof}
    for seed in SEEDS:
        skf = StratifiedKFold(NF, shuffle=True, random_state=seed)
        for f, (a, b) in enumerate(skf.split(X, y)):
            for name, m in clf_models(seed).items():
                t = time.time()
                m.fit(X.iloc[a], y[a], sample_weight=w[a] if use_w else None)
                oof[name][b] += m.predict_proba(X.iloc[b])[:, 1] / len(SEEDS)
                tst[name] += m.predict_proba(Xt)[:, 1] / (len(SEEDS) * NF)
            print(f"[{tag}] seed {seed} fold {f} done", flush=True)
    for k in oof:
        np.save(f"oof/p_{k}_{tag}.npy", oof[k]); np.save(f"oof/pt_{k}_{tag}.npy", tst[k])
        print(tag, k, "AUC %.4f AP %.4f LL %.5f" % (roc_auc_score(y, oof[k]),
              average_precision_score(y, oof[k]), log_loss(y, oof[k])))
    avg = np.mean([oof[k] for k in oof], 0)
    print(tag, "avg AUC %.4f AP %.4f" % (roc_auc_score(y, avg), average_precision_score(y, avg)))


def run_cost():
    pos = np.where(y == 1)[0]
    Xp, lc = X.iloc[pos], np.log(cost[pos])
    oof = np.zeros(len(pos)); tst_all = np.zeros(len(Xt)); trn_all = np.zeros(len(X))
    for seed in SEEDS:
        for a, b in KFold(NF, shuffle=True, random_state=seed).split(Xp):
            ms = [lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=7,
                                    min_child_samples=10, subsample=0.8, subsample_freq=1,
                                    colsample_bytree=0.5, reg_lambda=10, random_state=seed,
                                    verbosity=-1),
                  CatBoostRegressor(iterations=600, learning_rate=0.03, depth=4, l2_leaf_reg=10,
                                    random_seed=seed, verbose=False, allow_writing_files=False)]
            for m in ms:
                m.fit(Xp.iloc[a], lc[a])
                oof[b] += m.predict(Xp.iloc[b]) / (2 * len(SEEDS))
                tst_all += m.predict(Xt) / (2 * len(SEEDS) * NF)
                trn_all += m.predict(X) / (2 * len(SEEDS) * NF)
    # For non-failed train rows use the average of fold models (they never saw them).
    trn_all[pos] = oof
    np.save("oof/mu_train.npy", trn_all); np.save("oof/mu_test.npy", tst_all)
    r = np.corrcoef(oof, lc)[0, 1]
    print("cost model: corr(oof, log cost)=%.3f, RMSE %.3f vs const %.3f" % (
        r, np.sqrt(np.mean((oof - lc) ** 2)), lc.std()))


if __name__ == "__main__":
    what = sys.argv[1:] or ["cost", "w", "nw"]
    if "cost" in what: run_cost()
    if "w" in what: run_clf(True)
    if "nw" in what: run_clf(False)

In [ ]:
%run oof_models.py cost w

### Step 2: `full_fit.py`
Fit the ensemble on all training data; predict test both as-is and after quantile-matching each test sensor to the training distribution (removes the remaining depot-level scale shift in the heavy-usage tail).

In [ ]:
%%writefile full_fit.py
"""Fit the ensemble on all training data; predict test both as-is and after
quantile-matching each test sensor to the training distribution (removes the
remaining depot-level scale shift in the heavy-usage tail)."""
import numpy as np, pandas as pd, lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor
from fleet_prep import load, add_features

def qmatch(train_col, test_col):
    a = np.sort(train_col.dropna().to_numpy()); t = test_col.copy()
    m = t.notna()
    r = t[m].rank(pct=True, method="average").to_numpy()
    t[m] = np.quantile(a, np.clip(r - 0.5 / m.sum(), 0, 1))
    return t

tr, te, X, Xt = load()
sensors = [c for c in Xt.columns if c.startswith("s") and "_" not in c]
te_q = te.copy()
for c in sensors:
    te_q[c] = qmatch(tr[c], te[c])
Xq = add_features(te_q, sensors)
y = tr.failure.to_numpy(); cost = tr.repair_cost.fillna(0).to_numpy()
w = np.where((y == 0) & (tr.followup_days < 365), tr.followup_days / 365.0, 1.0)
P, PQ = [], []
for seed in [0, 1]:
    ms = [lgb.LGBMClassifier(n_estimators=600, learning_rate=0.02, num_leaves=15, min_child_samples=40,
                             subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=5,
                             random_state=seed, verbosity=-1),
          xgb.XGBClassifier(n_estimators=600, learning_rate=0.02, max_depth=4, min_child_weight=3,
                            subsample=0.8, colsample_bytree=0.5, reg_lambda=5, tree_method="hist",
                            random_state=seed, n_jobs=-1),
          CatBoostClassifier(iterations=1000, learning_rate=0.03, depth=5, l2_leaf_reg=8,
                             random_seed=seed, verbose=False, allow_writing_files=False)]
    for m in ms:
        m.fit(X, y, sample_weight=w)
        P.append(m.predict_proba(Xt)[:, 1]); PQ.append(m.predict_proba(Xq)[:, 1])
pos = y == 1
MU, MUQ = [], []
for seed in [0, 1]:
    for m in [lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=7, min_child_samples=10,
                                subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=10,
                                random_state=seed, verbosity=-1),
              CatBoostRegressor(iterations=600, learning_rate=0.03, depth=4, l2_leaf_reg=10,
                                random_seed=seed, verbose=False, allow_writing_files=False)]:
        m.fit(X[pos], np.log(cost[pos])); MU.append(m.predict(Xt)); MUQ.append(m.predict(Xq))
np.save("oof/full_pt.npy", np.mean(P, 0)); np.save("oof/full_ptq.npy", np.mean(PQ, 0))
np.save("oof/full_mut.npy", np.mean(MU, 0)); np.save("oof/full_mutq.npy", np.mean(MUQ, 0))
print("mean test p: as-is %.4f  quantile-matched %.4f  (train rate %.4f)" % (np.mean(P), np.mean(PQ), y.mean()))

In [ ]:
%run full_fit.py

### Step 3: `make_risk_variant.py`
Write a submission from the full-data ensemble with a given test-depot risk multiplier. Usage: python make_risk_variant.py <alpha> <name> [budget_mult]

In [ ]:
%%writefile make_risk_variant.py
"""Write a submission from the full-data ensemble with a given test-depot risk multiplier.
Usage: python make_risk_variant.py <alpha> <name> [budget_mult]"""
import sys
import numpy as np, pandas as pd
from policy import decide

alpha, name = float(sys.argv[1]), sys.argv[2]
tr = pd.read_csv("data/train.csv", usecols=["failure", "repair_cost"])
te = pd.read_csv("data/test.csv", usecols=["id"])
y = tr.failure.to_numpy(); c = tr.repair_cost.fillna(0).to_numpy()
mu = np.load("oof/mu_train.npy"); eps = np.log(c[y == 1]) - mu[y == 1]
bm = float(sys.argv[3]) if len(sys.argv) > 3 else 1.0
p = np.load("oof/full_pt.npy")
f, b = decide(p, np.load("oof/full_mut.npy"), eps, alpha, 1.0, 0.0, bm)
sub = pd.DataFrame({"id": te.id, "flag": f.astype(int), "budget": np.round(b, 2)})
assert len(sub) == 20000 and sub.flag.isin([0, 1]).all() and (sub.budget >= 0).all()
sub.to_csv(f"submission_{name}.csv", index=False)
print(name, "alpha", alpha, "budget_mult", bm, "inspections", int(f.sum()))

In [ ]:
%run make_risk_variant.py 6 v9_risk6

### Step 4: `full_fit_shapes.py`
Full-data ensemble (LightGBM + XGBoost + CatBoost, 3 seeds) with histogram shape features (hist_features.py). Saves oof/full_pt_shapes.npy and oof/full_mut_shapes.npy.

In [ ]:
%%writefile full_fit_shapes.py
"""Full-data ensemble (LightGBM + XGBoost + CatBoost, 3 seeds) with histogram shape
features (hist_features.py). Saves oof/full_pt_shapes.npy and oof/full_mut_shapes.npy."""
import numpy as np, lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor
from hist_features import with_shapes

tr, te, X, Xt = with_shapes()
y = tr.failure.to_numpy(); cost = tr.repair_cost.fillna(0).to_numpy()
w = np.where((y == 0) & (tr.followup_days < 365), tr.followup_days / 365.0, 1.0)
P = []
for seed in [0, 1, 2]:
    for m in [lgb.LGBMClassifier(n_estimators=600, learning_rate=0.02, num_leaves=15, min_child_samples=40,
                                 subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=5,
                                 random_state=seed, verbosity=-1),
              xgb.XGBClassifier(n_estimators=600, learning_rate=0.02, max_depth=4, min_child_weight=3,
                                subsample=0.8, colsample_bytree=0.5, reg_lambda=5, tree_method="hist",
                                random_state=seed, n_jobs=-1),
              CatBoostClassifier(iterations=1000, learning_rate=0.03, depth=5, l2_leaf_reg=8,
                                 random_seed=seed, verbose=False, allow_writing_files=False)]:
        m.fit(X, y, sample_weight=w); P.append(m.predict_proba(Xt)[:, 1])
    print("seed", seed, "done", flush=True)
pos = y == 1
MU = []
for seed in [0, 1]:
    for m in [lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=7, min_child_samples=10,
                                subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=10,
                                random_state=seed, verbosity=-1),
              CatBoostRegressor(iterations=600, learning_rate=0.03, depth=4, l2_leaf_reg=10,
                                random_seed=seed, verbose=False, allow_writing_files=False)]:
        m.fit(X[pos], np.log(cost[pos])); MU.append(m.predict(Xt))
pt = np.mean(P, 0)
np.save("oof/full_pt_shapes.npy", pt); np.save("oof/full_mut_shapes.npy", np.mean(MU, 0))
old = np.load("oof/full_pt.npy")
print("mean test p %.4f (old %.4f); corr with old %.4f" % (pt.mean(), old.mean(), np.corrcoef(pt, old)[0, 1]))

In [ ]:
%run full_fit_shapes.py

### Step 5: `cost_shapes.py`
Repair-cost model with histogram shape features (the top cost drivers by importance). Same 5-fold x 3-seed CV as oof_models.run_cost; saves oof/mu_train_shapes.npy (OOF for failed vehicles, fold average elsewhere) and oof/mu_test_shapes_cv.npy.

In [ ]:
%%writefile cost_shapes.py
"""Repair-cost model with histogram shape features (the top cost drivers by importance).
Same 5-fold x 3-seed CV as oof_models.run_cost; saves oof/mu_train_shapes.npy
(OOF for failed vehicles, fold average elsewhere) and oof/mu_test_shapes_cv.npy."""
import numpy as np, lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from hist_features import with_shapes
from cv_eval import y, cost

_, _, X, Xt = with_shapes()
pos = np.where(y == 1)[0]; Xp = X.iloc[pos]; lc = np.log(cost[pos])
oof = np.zeros(len(pos)); trn = np.zeros(len(X)); tst = np.zeros(len(Xt))
for seed in [0, 1, 2]:
    for a, b in KFold(5, shuffle=True, random_state=seed).split(Xp):
        for m in [lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=7, min_child_samples=10,
                                    subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=10,
                                    random_state=seed, verbosity=-1),
                  CatBoostRegressor(iterations=600, learning_rate=0.03, depth=4, l2_leaf_reg=10,
                                    random_seed=seed, verbose=False, allow_writing_files=False)]:
            m.fit(Xp.iloc[a], lc[a])
            oof[b] += m.predict(Xp.iloc[b]) / 6; trn += m.predict(X) / 30; tst += m.predict(Xt) / 30
trn[pos] = oof
base = np.load("oof/mu_train.npy")[pos]
rm = lambda v: np.sqrt(np.mean((v - lc) ** 2))
print("cost model RMSE: old %.4f (corr %.4f) -> shapes %.4f (corr %.4f)" % (rm(base), np.corrcoef(base, lc)[0, 1], rm(oof), np.corrcoef(oof, lc)[0, 1]))
np.save("oof/mu_train_shapes.npy", trn); np.save("oof/mu_test_shapes_cv.npy", tst)

In [ ]:
%run cost_shapes.py

### Step 6: `make_v30.py`
v30: shape features in BOTH models.

In [ ]:
%%writefile make_v30.py
"""v30: shape features in BOTH models.

Leaderboard calibration (after v27) puts ~44% of v9's cost in budget error on vehicles
that do fail, i.e. in repair-cost uncertainty. The histogram shape features turn out to
be the strongest repair-cost drivers (B_f1, hist_mean_bin, C_mean, C_f8, ...), but the
cost model never had them.

  cost model RMSE (log, OOF on 397 failures): 0.807 -> 0.724 (corr 0.872 -> 0.898)
  OOF competition cost (LGB+Cat): old/old 46,810; new classifier only -3.3%; new cost only
  -1.5%; both -4.9%.
  Shift check: on comparable vehicles the cost prediction moves the same amount on train
  (p > 0.1: mean |delta| 0.214) and test (v29-flagged: 0.228), so no extrapolation.

v30  classifier and cost model with shape features, v9's risk x6 policy
v31  v9's classifier + new cost model (isolates the cost-model gain)
"""
import numpy as np, pandas as pd
from policy import decide
from cv_eval import y, cost

te = pd.read_csv("data/test.csv", usecols=["id"])
mu_new = np.load("oof/mu_train_shapes.npy"); eps_new = np.log(cost[y == 1]) - mu_new[y == 1]
mut_new = np.load("oof/full_mut_shapes.npy")
f9 = pd.read_csv("submission_v9_risk6.csv").flag.to_numpy()


def write(name, p):
    f, b = decide(p, mut_new, eps_new, 6.0, 1.0, 0.0)
    sub = pd.DataFrame({"id": te.id, "flag": f.astype(int), "budget": np.round(b, 2)})
    assert len(sub) == 20000 and sub.flag.isin([0, 1]).all() and (sub.budget >= 0).all()
    assert (sub.loc[sub.flag == 0, "budget"] == 0).all()
    sub.to_csv(f"submission_{name}.csv", index=False)
    print(f"{name}: inspections {f.sum()}, mean budget {b[f == 1].mean():.0f}, shared with v9 {(f & f9).sum()}")


write("v30_shapes_both", np.load("oof/full_pt_shapes.npy"))
write("v31_v9clf_newcost", np.load("oof/full_pt.npy"))

In [ ]:
%run make_v30.py

### Step 7: `hist_features.py`
Five more 10-bin histogram groups (found by non-negative regression on the s054-s063 total): s023-s032, s037-s046, s080-s089, s129-s138, s154-s163. Each group's bins sum to a fixed multiple of the same operating total. None is shifted in test (bin shares match).

In [ ]:
%%writefile hist_features.py
"""Five more 10-bin histogram groups (found by non-negative regression on the s054-s063
total): s023-s032, s037-s046, s080-s089, s129-s138, s154-s163. Each group's bins sum to
a fixed multiple of the same operating total. None is shifted in test (bin shares match).

Trees see raw bin counts, which mostly encode the total (age/usage). Shape features
(bin shares, mean bin, spread, tail share) describe *how* the vehicle is used.
"""
import sys, numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from fleet_prep import load
from cv_eval import y, cost, weights
from policy import decide, competition_cost

GROUPS = {"A": (23, 32), "B": (37, 46), "C": (80, 89), "D": (129, 138), "E": (154, 163)}


def shape_features(df):
    out = {}
    for k, (a, b) in GROUPS.items():
        h = df[[f"s{i:03d}" for i in range(a, b + 1)]].to_numpy(float)
        tot = np.nansum(h, 1); tot[tot == 0] = np.nan
        fr = h / tot[:, None]
        w = np.arange(10)
        mb = np.nansum(fr * w, 1)
        out.update({f"{k}_f{j}": fr[:, j] for j in range(10)})
        out[f"{k}_mean"] = mb
        out[f"{k}_sd"] = np.sqrt(np.nansum(fr * (w - mb[:, None]) ** 2, 1))
        out[f"{k}_hi"] = np.nansum(fr[:, 7:], 1)
        out[f"{k}_lo"] = np.nansum(fr[:, :2], 1)
    return pd.DataFrame(out, index=df.index)


def with_shapes():
    tr, te, X, Xt = load()
    return tr, te, pd.concat([X, shape_features(tr)], axis=1), pd.concat([Xt, shape_features(te)], axis=1)


if __name__ == "__main__":
    mu = np.load("oof/mu_train.npy"); eps = np.log(cost[y == 1]) - mu[y == 1]
    _, _, X0, _ = load(); _, _, X1, _ = with_shapes()
    for name, X in [("baseline", X0), ("+shapes", X1)]:
        oof = np.zeros(len(X))
        for seed in [0, 1]:
            for a, b in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y):
                m = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.02, num_leaves=15, min_child_samples=40,
                                       subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=5,
                                       random_state=seed, verbosity=-1).fit(X.iloc[a], y[a], sample_weight=weights[a])
                oof[b] += m.predict_proba(X.iloc[b])[:, 1] / 2
        c = min(competition_cost(cost, *decide(oof, mu, eps, al, 1.0, 0.0)).mean() for al in [1, 1.25, 1.5, 2])
        print(f"{name:9s} AUC {roc_auc_score(y, oof):.4f} AP {average_precision_score(y, oof):.4f} OOF cost {c:.4f}", flush=True)
        np.save(f"oof/lgb2_{name.strip('+')}.npy", oof)

In [ ]:
%run hist_features.py

### Step 8: `cat_shapes.py`


In [ ]:
%%writefile cat_shapes.py
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from fleet_prep import load
from hist_features import with_shapes
from cv_eval import y, cost, weights
from policy import decide, competition_cost
mu = np.load("oof/mu_train.npy"); eps = np.log(cost[y == 1]) - mu[y == 1]
_, _, X0, _ = load(); _, _, X1, _ = with_shapes()
res = {}
for name, X in [("baseline", X0), ("shapes", X1)]:
    oof = np.zeros(len(X))
    for seed in [0, 1]:
        for a, b in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y):
            m = CatBoostClassifier(iterations=1000, learning_rate=0.03, depth=5, l2_leaf_reg=8, random_seed=seed,
                                   verbose=False, allow_writing_files=False, thread_count=-1)
            m.fit(X.iloc[a], y[a], sample_weight=weights[a]); oof[b] += m.predict_proba(X.iloc[b])[:, 1] / 2
    res[name] = oof
    c = min(competition_cost(cost, *decide(oof, mu, eps, al, 1.0, 0.0)).mean() for al in [1, 1.25, 1.5, 2])
    print(f"cat {name:9s} AUC {roc_auc_score(y, oof):.4f} AP {average_precision_score(y, oof):.4f} OOF cost {c:.4f}", flush=True)
    np.save(f"oof/cat2_{name}.npy", oof)
for name in ["baseline", "shapes"]:
    lg = np.load(f"oof/lgb2_{name}.npy"); avg = (lg + res[name]) / 2
    c = min(competition_cost(cost, *decide(avg, mu, eps, al, 1.0, 0.0)).mean() for al in [1, 1.25, 1.5, 2])
    print(f"lgb+cat {name:9s} AUC {roc_auc_score(y, avg):.4f} AP {average_precision_score(y, avg):.4f} OOF cost {c:.4f}")

In [ ]:
%run cat_shapes.py

### Step 9: `budget_blend.py`
Budgets from a tree + smooth (spline-ridge on bounded shape features) cost blend; flags keep the v30 tree cost model. Grid over blend weight and spline settings.

In [ ]:
%%writefile budget_blend.py
"""Budgets from a tree + smooth (spline-ridge on bounded shape features) cost blend; flags
keep the v30 tree cost model. Grid over blend weight and spline settings."""
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, SplineTransformer
from cost_linear_shapes import F, Ft, pos, lc, tree, tree_t, P
from cv_eval import y, cost
from policy import decide, competition_cost


def spline_cv(knots, deg):
    oof = np.zeros(len(pos)); trn = np.zeros(len(F)); tst = np.zeros(len(Ft))
    for seed in [0, 1, 2]:
        for a, b in KFold(5, shuffle=True, random_state=seed).split(pos):
            m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                              SplineTransformer(n_knots=knots, degree=deg), RidgeCV(np.logspace(-1, 4, 25)))
            m.fit(F.iloc[pos[a]], lc[a])
            oof[b] += m.predict(F.iloc[pos[b]]) / 3; trn += m.predict(F) / 15; tst += m.predict(Ft) / 15
    trn[pos] = oof
    return trn, tst


def score(mu_b):
    best = None
    for al in [1, 1.25, 1.5]:
        f, _ = decide(P, tree, lc - tree[pos], al, 1.0, 0.0)
        _, b = decide(P, mu_b, lc - mu_b[pos], al, 1.0, -1e9)
        c = competition_cost(cost, f, np.where(f == 1, b, 0)).sum()
        best = c if best is None or c < best else best
    return best


if __name__ == "__main__":
    for knots, deg in [(4, 2), (5, 3), (3, 1)]:
        trn, tst = spline_cv(knots, deg)
        print(f"spline knots {knots} deg {deg}: alone RMSE {np.sqrt(np.mean((trn[pos] - lc) ** 2)):.4f}")
        for w in [0.5, 0.65, 0.8]:
            mu = (1 - w) * tree + w * trn
            print(f"   w{w}: RMSE {np.sqrt(np.mean((mu[pos] - lc) ** 2)):.4f} OOF cost {score(mu):.0f}", flush=True)
            np.save(f"oof/bb_k{knots}d{deg}_w{w}_train.npy", mu); np.save(f"oof/bb_k{knots}d{deg}_w{w}_test.npy", (1 - w) * tree_t + w * tst)
    print("v30 reference 44517")

In [ ]:
%run budget_blend.py

### Step 10: `make_v33.py`
v33: v30's inspections unchanged; budgets from a blend of the v30 tree cost model and a piecewise-linear spline ridge on bounded histogram-shape features (budget_blend.py, knots 3, degree 1, weight 0.5).

In [ ]:
%%writefile make_v33.py
"""v33: v30's inspections unchanged; budgets from a blend of the v30 tree cost model and a
piecewise-linear spline ridge on bounded histogram-shape features (budget_blend.py,
knots 3, degree 1, weight 0.5).

CV: cost RMSE 0.724 -> 0.696; OOF competition cost 44,517 -> 43,943 (-1.3%) with flags
unchanged. Bounded inputs (shares in [0, 1]) cannot extrapolate; shift check below.
"""
import numpy as np, pandas as pd
from policy import decide
from cv_eval import y, cost

te = pd.read_csv("data/test.csv", usecols=["id"])
pos = y == 1; lc = np.log(cost[pos])
mu_b = np.load("oof/bb_k3d1_w0.5_train.npy"); mut_b = np.load("oof/bb_k3d1_w0.5_test.npy")
tree = np.load("oof/mu_train_shapes.npy"); tree_t = np.load("oof/full_mut_shapes.npy")
pt = np.load("oof/full_pt_shapes.npy")
P = (np.load("oof/lgb2_shapes.npy") + np.load("oof/cat2_shapes.npy")) / 2
print("shift check mean |budget-model change|: train p>0.1 %.3f | test p>0.1 %.3f"
      % (np.abs(mu_b - tree)[P > 0.1].mean(), np.abs(mut_b - tree_t)[pt > 0.1].mean()))
f30 = pd.read_csv("submission_v30_shapes_both.csv").flag.to_numpy()
_, b = decide(pt, mut_b, lc - mu_b[pos], 6.0, 1.0, -1e9)
b = np.where(f30 == 1, b, 0.0)
sub = pd.DataFrame({"id": te.id, "flag": f30, "budget": np.round(b, 2)})
assert len(sub) == 20000 and (sub.loc[sub.flag == 0, "budget"] == 0).all() and (sub.budget >= 0).all()
sub.to_csv("submission_v33_budget_blend.csv", index=False)
b30 = pd.read_csv("submission_v30_shapes_both.csv").budget.to_numpy()
print(f"v33: inspections {f30.sum()} (same as v30), mean budget {b[f30 == 1].mean():.0f} (v30 {b30[f30 == 1].mean():.0f}), "
      f"median |budget change| {np.median(np.abs(b - b30)[f30 == 1]):.0f}")

In [ ]:
%run make_v33.py

### Step 11: `make_v32.py`
v32: v30 with risk x7 instead of x6. v30 inspects 1632 vehicles (v9: 1674); on v9's models the leaderboard optimum was ~1690 inspections, so v32 brings v30's count back up.

In [ ]:
%%writefile make_v32.py
"""v32: v30 with risk x7 instead of x6. v30 inspects 1632 vehicles (v9: 1674); on v9's
models the leaderboard optimum was ~1690 inspections, so v32 brings v30's count back up."""
import numpy as np, pandas as pd
from policy import decide
from cv_eval import y, cost

te = pd.read_csv("data/test.csv", usecols=["id"])
mu = np.load("oof/mu_train_shapes.npy"); eps = np.log(cost[y == 1]) - mu[y == 1]
f, b = decide(np.load("oof/full_pt_shapes.npy"), np.load("oof/full_mut_shapes.npy"), eps, 7.0, 1.0, 0.0)
sub = pd.DataFrame({"id": te.id, "flag": f.astype(int), "budget": np.round(b, 2)})
assert len(sub) == 20000 and (sub.loc[sub.flag == 0, "budget"] == 0).all()
sub.to_csv("submission_v32_shapes_risk7.csv", index=False)
f30 = pd.read_csv("submission_v30_shapes_both.csv").flag.to_numpy()
print(f"v32: inspections {f.sum()}, mean budget {b[f == 1].mean():.0f}, v30 flags kept {(f & f30).sum()} of {f30.sum()}")

In [ ]:
%run make_v32.py

### Step 12

In [ ]:
# Spline-only cost model (blend weight 1.0) used for v34's budgets
import numpy as np
from budget_blend import spline_cv
trn, tst = spline_cv(3, 1)
np.save("oof/bb_k3d1_w1.0_train.npy", trn); np.save("oof/bb_k3d1_w1.0_test.npy", tst)

### Step 13: `make_v34_v35.py`
v34-v35, after v33 (16,395.84) beat v30 by 479 points, more than CV predicted (-1.3%): the smooth cost model generalises to the test depot better than the trees.

In [ ]:
%%writefile make_v34_v35.py
"""v34-v35, after v33 (16,395.84) beat v30 by 479 points, more than CV predicted (-1.3%):
the smooth cost model generalises to the test depot better than the trees.

v34  v30 inspections; budgets from the spline ridge on bounded shape features ALONE
     (blend weight 1.0; OOF cost 43,843 vs v33's 43,943; shift check train 0.285 / test 0.284)
v35  v33 budgets (weight 0.5) on v32's inspections (risk x7, 1696 vehicles)
"""
import numpy as np, pandas as pd
from policy import decide
from cv_eval import y, cost

te = pd.read_csv("data/test.csv", usecols=["id"])
pos = y == 1; lc = np.log(cost[pos])
pt = np.load("oof/full_pt_shapes.npy")


def write(name, flags, w, alpha):
    mu_b = np.load(f"oof/bb_k3d1_w{w}_train.npy"); mut_b = np.load(f"oof/bb_k3d1_w{w}_test.npy")
    _, b = decide(pt, mut_b, lc - mu_b[pos], alpha, 1.0, -1e9)
    b = np.where(flags == 1, b, 0.0)
    sub = pd.DataFrame({"id": te.id, "flag": flags.astype(int), "budget": np.round(b, 2)})
    assert len(sub) == 20000 and (sub.loc[sub.flag == 0, "budget"] == 0).all() and (sub.budget >= 0).all()
    sub.to_csv(f"submission_{name}.csv", index=False)
    b33 = pd.read_csv("submission_v33_budget_blend.csv").budget.to_numpy()
    print(f"{name}: inspections {flags.sum()}, mean budget {b[flags == 1].mean():.0f}")
    return b


f30 = pd.read_csv("submission_v30_shapes_both.csv").flag.to_numpy()
f32 = pd.read_csv("submission_v32_shapes_risk7.csv").flag.to_numpy()
write("v34_budget_smooth", f30, 1.0, 6.0)
write("v35_v33_risk7", f32, 0.5, 7.0)

In [ ]:
%run make_v34_v35.py

### Step 14: `smooth_cost3.py`
Stability of the interaction gain: number of interacting features, clipping to the observed log-cost range, and blending with the v34 spline.

In [ ]:
%%writefile smooth_cost3.py
"""Stability of the interaction gain: number of interacting features, clipping to the
observed log-cost range, and blending with the v34 spline."""
import numpy as np, pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, SplineTransformer, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from smooth_cost2 import cv, score, F, Ft, pos, lc
import smooth_cost2 as sc

corr = pd.Series({c: abs(np.corrcoef(F.iloc[pos][c].fillna(F[c].median()), lc)[0, 1]) for c in F.columns}).sort_values(ascending=False)
LO, HI = np.log(5), np.log(8000)


def inter(k):
    top = [F.columns.get_loc(c) for c in corr.index[:k]]
    return lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                 ColumnTransformer([("spl", SplineTransformer(n_knots=3, degree=1), list(range(F.shape[1]))),
                                                    ("int", PolynomialFeatures(2, interaction_only=True, include_bias=False), top)]),
                                 RidgeCV(np.logspace(-1, 4, 25)))


if __name__ == "__main__":
    base = np.load("oof/sc2_v34_spline_k3_d1_train.npy")
    for k in [8, 10, 15, 20, 25]:
        trn, tst = cv(f"interactions top{k}", inter(k))
        c = score(np.clip(trn, LO, HI)); cb = score(np.clip(0.5 * trn + 0.5 * base, LO, HI))
        print(f"     clipped {c:.0f} | 50/50 with v34 spline, clipped {cb:.0f}", flush=True)

In [ ]:
%run smooth_cost3.py

### Step 15: `make_v36_v37.py`
v36-v37: budgets from a smooth cost model with pairwise interactions among the 8 bounded shape features most correlated with log repair cost (smooth_cost3.py), clipped to the observed cost range [5, 8000]. Inspections unchanged.

In [ ]:
%%writefile make_v36_v37.py
"""v36-v37: budgets from a smooth cost model with pairwise interactions among the 8 bounded
shape features most correlated with log repair cost (smooth_cost3.py), clipped to the
observed cost range [5, 8000]. Inspections unchanged.

Why it helps: the score charges 0.20 per unit of shortfall, so what matters is the
expensive tail. With interactions the model gives expensive failures higher budgets
(OOF shortfall on failing vehicles 13,915 -> ~11,600). CV: OOF cost 43,843 (v34) -> 42,096
(-4.0%); every variant from 8 to 25 interacting features is 2-4.6% better.

v36  v30/v34 inspections (1632)
v37  v32/v35 inspections (risk x7, 1696)
Budgets are capped at 8000, the largest repair cost in training (two vehicles sit exactly there).
"""
import numpy as np, pandas as pd
from policy import decide
from cv_eval import y, cost

te = pd.read_csv("data/test.csv", usecols=["id"])
pos = y == 1; lc = np.log(cost[pos]); LO, HI = np.log(5), np.log(8000)
mu_b = np.clip(np.load("oof/sc2_interactions_top8_train.npy"), LO, HI)
mut_b = np.clip(np.load("oof/sc2_interactions_top8_test.npy"), LO, HI)
pt = np.load("oof/full_pt_shapes.npy")
PT = (np.load("oof/lgb2_shapes.npy") + np.load("oof/cat2_shapes.npy")) / 2
tree, tree_t = np.load("oof/mu_train_shapes.npy"), np.load("oof/full_mut_shapes.npy")
print("shift check |budget model - tree|: train p>0.1 %.3f | test p>0.1 %.3f"
      % (np.abs(mu_b - tree)[PT > 0.1].mean(), np.abs(mut_b - tree_t)[pt > 0.1].mean()))
b34 = pd.read_csv("submission_v34_budget_smooth.csv").budget.to_numpy()


def write(name, flags, alpha):
    _, b = decide(pt, mut_b, lc - mu_b[pos], alpha, 1.0, -1e9)
    b = np.where(flags == 1, np.minimum(b, 8000.0), 0.0)  # 8000 = largest repair cost seen (a hard cap)
    sub = pd.DataFrame({"id": te.id, "flag": flags.astype(int), "budget": np.round(b, 2)})
    assert len(sub) == 20000 and (sub.loc[sub.flag == 0, "budget"] == 0).all() and (sub.budget >= 0).all()
    sub.to_csv(f"submission_{name}.csv", index=False)
    print(f"{name}: inspections {flags.sum()}, mean budget {b[flags == 1].mean():.0f}, max {b.max():.0f}")


write("v36_budget_interactions", pd.read_csv("submission_v30_shapes_both.csv").flag.to_numpy(), 6.0)
write("v37_interactions_risk7", pd.read_csv("submission_v32_shapes_risk7.csv").flag.to_numpy(), 7.0)
print(f"(v34 mean budget {b34[b34 > 0].mean():.0f})")

In [ ]:
%run make_v36_v37.py

## Output files

In [ ]:
import glob, pandas as pd
for p in sorted(glob.glob("submission_*.csv")):
    s = pd.read_csv(p)
    print(f"{p}: {int(s.flag.sum())} inspections, mean budget {s.budget[s.flag == 1].mean():.0f}")

## Other scripts in this version
Experiments and tuning scripts that are not needed for this version's submission. Shown for completeness; they are written to disk but not run.

### `clf_search.py`
Classifier variants on top of shape features, scored by OOF competition cost with the v30 cost model (LightGBM, 5-fold x 2 seeds, same splits as hist_features.py).

In [ ]:
%%writefile clf_search.py
"""Classifier variants on top of shape features, scored by OOF competition cost with the
v30 cost model (LightGBM, 5-fold x 2 seeds, same splits as hist_features.py)."""
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from hist_features import with_shapes
from cost_search2 import rates
from cv_eval import y, cost, weights
from policy import decide, competition_cost

_, _, X, _ = with_shapes()
mu = np.load("oof/mu_train_shapes.npy"); eps = np.log(cost[y == 1]) - mu[y == 1]
shape_cols = [c for c in X.columns if c[:2] in ("A_", "B_", "C_", "D_", "E_")]


def run(name, D, **k):
    p = dict(n_estimators=600, learning_rate=0.02, num_leaves=15, min_child_samples=40, subsample=0.8,
             subsample_freq=1, colsample_bytree=0.5, reg_lambda=5, verbosity=-1); p.update(k)
    oof = np.zeros(len(D))
    for seed in [0, 1]:
        for a, b in StratifiedKFold(5, shuffle=True, random_state=seed).split(D, y):
            m = lgb.LGBMClassifier(random_state=seed, **p).fit(D.iloc[a], y[a], sample_weight=weights[a])
            oof[b] += m.predict_proba(D.iloc[b])[:, 1] / 2
    c = min(competition_cost(cost, *decide(oof, mu, eps, al, 1.0, 0.0)).sum() for al in [1, 1.25, 1.5])
    print(f"{name:28s} AUC {roc_auc_score(y, oof):.4f} AP {average_precision_score(y, oof):.4f} OOF cost {c:7.0f}", flush=True)
    np.save("oof/clf_" + "".join(ch if ch.isalnum() else "_" for ch in name) + ".npy", oof)


run("shapes (v30 lgb)", X)
run("shapes + rates", pd.concat([X, rates(X)], axis=1))
run("shapes, colsample 0.3", X, colsample_bytree=0.3)
run("shapes, 31 leaves", X, num_leaves=31, min_child_samples=60)

### `cost_search.py`
Search for a better conditional repair-cost model (log cost of failed vehicles). Compares OOF RMSE on the 397 failed vehicles, same 5-fold x 3-seed splits.

In [ ]:
%%writefile cost_search.py
"""Search for a better conditional repair-cost model (log cost of failed vehicles).
Compares OOF RMSE on the 397 failed vehicles, same 5-fold x 3-seed splits."""
import numpy as np, lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from fleet_prep import load
from cv_eval import y, cost

_, _, X, Xt = load()
pos = np.where(y == 1)[0]; Xp = X.iloc[pos].reset_index(drop=True); lc = np.log(cost[pos])
base = np.load("oof/mu_train.npy")[pos]
print("baseline RMSE %.4f corr %.4f" % (np.sqrt(np.mean((base - lc) ** 2)), np.corrcoef(base, lc)[0, 1]))
Xs = np.sign(Xp) * np.log1p(np.abs(Xp))

def cv(make, data, name):
    oof = np.zeros(len(pos))
    for seed in [0, 1, 2]:
        for a, b in KFold(5, shuffle=True, random_state=seed).split(data):
            m = make(seed).fit(data.iloc[a], lc[a]); oof[b] += m.predict(data.iloc[b]) / 3
    print(f"{name:30s} RMSE {np.sqrt(np.mean((oof - lc) ** 2)):.4f} corr {np.corrcoef(oof, lc)[0, 1]:.4f}", flush=True)
    np.save(f"oof/cs_{name}.npy", oof)
    return oof

o = {}
o["ridge"] = cv(lambda s: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), RidgeCV(np.logspace(-1, 4, 20))), Xs, "ridge")
o["et"] = cv(lambda s: make_pipeline(SimpleImputer(strategy="median"), ExtraTreesRegressor(500, min_samples_leaf=3, max_features=0.3, n_jobs=-1, random_state=s)), Xp, "et")
o["rf"] = cv(lambda s: make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(500, min_samples_leaf=3, max_features=0.3, n_jobs=-1, random_state=s)), Xp, "rf")
o["lgb_small"] = cv(lambda s: lgb.LGBMRegressor(n_estimators=600, learning_rate=0.01, num_leaves=4, min_child_samples=8, subsample=0.7, subsample_freq=1, colsample_bytree=0.3, reg_lambda=20, random_state=s, verbosity=-1), Xp, "lgb_small")
o["cat_d6"] = cv(lambda s: CatBoostRegressor(iterations=1500, learning_rate=0.02, depth=6, l2_leaf_reg=20, random_seed=s, verbose=False, allow_writing_files=False, thread_count=-1), Xp, "cat_d6")
for k, v in o.items():
    for wgt in [0.3, 0.5]:
        bl = (1 - wgt) * base + wgt * v
        print(f"blend base+{k} w{wgt}: RMSE {np.sqrt(np.mean((bl - lc) ** 2)):.4f}")

### `cost_search2.py`
Second repair-cost search, on top of the shape features (v30). Each variant is scored by OOF log-RMSE on the 397 failures AND by OOF competition cost with the v29/v30 classifier.

In [ ]:
%%writefile cost_search2.py
"""Second repair-cost search, on top of the shape features (v30). Each variant is scored by
OOF log-RMSE on the 397 failures AND by OOF competition cost with the v29/v30 classifier."""
import sys, numpy as np, pandas as pd, lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from hist_features import with_shapes
from cv_eval import y, cost
from policy import decide, competition_cost

_, _, X, Xt = with_shapes()
pos = np.where(y == 1)[0]; lc = np.log(cost[pos])
P = (np.load("oof/lgb2_shapes.npy") + np.load("oof/cat2_shapes.npy")) / 2


def rates(df):
    tot = df["hist_total"].replace(0, np.nan)
    cols = [c for c in df.columns if c.startswith("s") and "_" not in c]
    return pd.DataFrame({f"{c}_rate": df[c] / tot for c in cols}, index=df.index)


def lgbm(s, **k):
    p = dict(n_estimators=300, learning_rate=0.03, num_leaves=7, min_child_samples=10, subsample=0.8,
             subsample_freq=1, colsample_bytree=0.5, reg_lambda=10, random_state=s, verbosity=-1); p.update(k)
    return lgb.LGBMRegressor(**p)


def cat(s, **k):
    p = dict(iterations=600, learning_rate=0.03, depth=4, l2_leaf_reg=10, random_seed=s, verbose=False,
             allow_writing_files=False, thread_count=-1); p.update(k)
    return CatBoostRegressor(**p)


def run(name, D, makers):
    Dp = D.iloc[pos]; oof = np.zeros(len(pos)); trn = np.zeros(len(D))
    for seed in [0, 1, 2]:
        for a, b in KFold(5, shuffle=True, random_state=seed).split(Dp):
            for mk in makers:
                m = mk(seed).fit(Dp.iloc[a], lc[a])
                oof[b] += m.predict(Dp.iloc[b]) / (3 * len(makers)); trn += m.predict(D) / (15 * len(makers))
    trn[pos] = oof
    e = lc - oof
    c = min(competition_cost(cost, *decide(P, trn, e, al, 1.0, 0.0)).sum() for al in [1, 1.25, 1.5])
    print(f"{name:34s} RMSE {np.sqrt(np.mean(e ** 2)):.4f}  OOF cost {c:7.0f}", flush=True)
    np.save("oof/cs2_" + "".join(ch if ch.isalnum() else "_" for ch in name) + ".npy", trn)
    return c


shape_cols = [c for c in X.columns if c[:2] in ("A_", "B_", "C_", "D_", "E_") or c.startswith("hist") or c.endswith("_frac")]
XR = pd.concat([X, rates(X)], axis=1)
if __name__ == "__main__": run("base (v30)", X, [lgbm, cat])
if __name__ == "__main__": run("shapes-only features", X[shape_cols + ["n_missing", "n_zero"]], [lgbm, cat])
if __name__ == "__main__": run("cat only", X, [cat])
if __name__ == "__main__": run("cat depth6 1200it", X, [lambda s: cat(s, depth=6, iterations=1200, learning_rate=0.02)])
if __name__ == "__main__": run("lgb+cat slower/more", X, [lambda s: lgbm(s, n_estimators=800, learning_rate=0.01), lambda s: cat(s, iterations=1500, learning_rate=0.015)])
if __name__ == "__main__": run("+ usage rates", XR, [lgbm, cat])
if __name__ == "__main__": run("cat huber", X, [lambda s: cat(s, loss_function="Huber:delta=1.0")])

### `full_fit_more.py`
More full-data bagging for the v9 ranking (v9 used seeds 0-1 of full_fit.py), plus a ridge component in the repair-cost model (CV RMSE 0.807 -> 0.790).

In [ ]:
%%writefile full_fit_more.py
"""More full-data bagging for the v9 ranking (v9 used seeds 0-1 of full_fit.py),
plus a ridge component in the repair-cost model (CV RMSE 0.807 -> 0.790).

Saves oof/full_pt5.npy (mean over seeds 0-4, 15 classifiers) and
oof/full_mut_ridge.npy (0.7 * tree cost model + 0.3 * ridge on signed-log features).
"""
import numpy as np, lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from fleet_prep import load

tr, te, X, Xt = load()
y = tr.failure.to_numpy(); cost = tr.repair_cost.fillna(0).to_numpy()
w = np.where((y == 0) & (tr.followup_days < 365), tr.followup_days / 365.0, 1.0)
P = []
for seed in [2, 3, 4]:
    for m in [lgb.LGBMClassifier(n_estimators=600, learning_rate=0.02, num_leaves=15, min_child_samples=40,
                                 subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=5,
                                 random_state=seed, verbosity=-1),
              xgb.XGBClassifier(n_estimators=600, learning_rate=0.02, max_depth=4, min_child_weight=3,
                                subsample=0.8, colsample_bytree=0.5, reg_lambda=5, tree_method="hist",
                                random_state=seed, n_jobs=-1),
              CatBoostClassifier(iterations=1000, learning_rate=0.03, depth=5, l2_leaf_reg=8,
                                 random_seed=seed, verbose=False, allow_writing_files=False)]:
        m.fit(X, y, sample_weight=w); P.append(m.predict_proba(Xt)[:, 1])
    print("seed", seed, "done", flush=True)
pt5 = (2 * np.load("oof/full_pt.npy") + 3 * np.mean(P, 0)) / 5
np.save("oof/full_pt5.npy", pt5)
print("corr with v9 probabilities %.4f" % np.corrcoef(pt5, np.load("oof/full_pt.npy"))[0, 1])

pos = y == 1
sl = lambda d: np.sign(d) * np.log1p(np.abs(d))
ridge = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), RidgeCV(np.logspace(-1, 4, 20)))
ridge.fit(sl(X[pos]), np.log(cost[pos]))
mr = 0.7 * np.load("oof/full_mut.npy") + 0.3 * ridge.predict(sl(Xt))
np.save("oof/full_mut_ridge.npy", mr)
print("cost model change: mean |delta mu| %.3f" % np.abs(mr - np.load("oof/full_mut.npy")).mean())

### `heavy_group.py`
Where the failures are: ~77% of training failures sit in the heavy-usage group (mean usage percentile > 0.8, 4.5% of train, 6.8% of test). Does a model trained only on that group rank vehicles inside it better than the global ensemble?

In [ ]:
%%writefile heavy_group.py
"""Where the failures are: ~77% of training failures sit in the heavy-usage group
(mean usage percentile > 0.8, 4.5% of train, 6.8% of test). Does a model trained
only on that group rank vehicles inside it better than the global ensemble?"""
import numpy as np, lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from fleet_prep import load
from cv_eval import y, weights

_, _, X, Xt = load()
r = np.load("oof/usage_rank_train.npy")
g = np.where(r > 0.75)[0]
p = np.mean([np.load(f"oof/p_{n}_w.npy") for n in ("lgb", "xgb", "cat")], 0)
print(f"group size {len(g)}, failures {y[g].sum()} of {y.sum()}")
print("global ensemble inside group: AUC %.4f AP %.4f" % (roc_auc_score(y[g], p[g]), average_precision_score(y[g], p[g])))
Xg, yg, wg = X.iloc[g], y[g], weights[g]
oof = np.zeros(len(g))
for seed in [0, 1, 2]:
    for a, b in StratifiedKFold(5, shuffle=True, random_state=seed).split(Xg, yg):
        ms = [lgb.LGBMClassifier(n_estimators=400, learning_rate=0.02, num_leaves=7, min_child_samples=20,
                                 subsample=0.8, subsample_freq=1, colsample_bytree=0.4, reg_lambda=10,
                                 random_state=seed, verbosity=-1),
              CatBoostClassifier(iterations=800, learning_rate=0.03, depth=4, l2_leaf_reg=10, random_seed=seed,
                                 verbose=False, allow_writing_files=False, thread_count=-1)]
        for m in ms:
            m.fit(Xg.iloc[a], yg[a], sample_weight=wg[a]); oof[b] += m.predict_proba(Xg.iloc[b])[:, 1] / 6
print("group-only model:             AUC %.4f AP %.4f" % (roc_auc_score(yg, oof), average_precision_score(yg, oof)))
from scipy.stats import rankdata
for wt in [0.3, 0.5]:
    bl = (1 - wt) * rankdata(p[g]) + wt * rankdata(oof)
    print(f"rank blend w{wt}:              AUC %.4f AP %.4f" % (roc_auc_score(yg, bl), average_precision_score(yg, bl)))
np.save("oof/group_oof.npy", oof); np.save("oof/group_idx.npy", g)

### `logit_model.py`
Logistic regression on log1p sensors. Unlike trees it extrapolates, which matters because the test depot's usage tail goes 1.3-1.8x beyond the training range.

In [ ]:
%%writefile logit_model.py
"""Logistic regression on log1p sensors. Unlike trees it extrapolates, which matters
because the test depot's usage tail goes 1.3-1.8x beyond the training range."""
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from fleet_prep import load

tr, te, X, Xt = load()
y = tr.failure.to_numpy()
w = np.where((y == 0) & (tr.followup_days < 365), tr.followup_days / 365.0, 1.0)
cols = [c for c in X.columns if not c.endswith("_frac")]
sl = lambda a: np.sign(a) * np.log1p(np.abs(a))

def model(C):
    return make_pipeline(FunctionTransformer(sl), SimpleImputer(strategy="median", add_indicator=True),
                         StandardScaler(), LogisticRegression(C=C, max_iter=5000))

for C in [0.003, 0.01, 0.03, 0.1]:
    oof = np.zeros(len(X))
    for a, b in StratifiedKFold(5, shuffle=True, random_state=0).split(X, y):
        m = model(C).fit(X[cols].iloc[a], y[a], logisticregression__sample_weight=w[a])
        oof[b] = m.predict_proba(X[cols].iloc[b])[:, 1]
    print(f"C={C}: AUC {roc_auc_score(y, oof):.4f} AP {average_precision_score(y, oof):.4f}")
    np.save(f"oof/p_lr_C{C}.npy", oof)
    m = model(C).fit(X[cols], y, logisticregression__sample_weight=w)
    np.save(f"oof/pt_lr_C{C}.npy", m.predict_proba(Xt[cols])[:, 1])

### `make_grid.py`
v16-v23: one-knob variants around v14, for leaderboard comparison of global settings.

In [ ]:
%%writefile make_grid.py
"""v16-v23: one-knob variants around v14, for leaderboard comparison of global settings.

Each file changes exactly ONE setting relative to v14 (1692 inspections, EM prior,
heteroscedastic budgets, bagged probabilities), so a leaderboard score tells us
which direction that setting should move. No per-vehicle label probing: every
change is a single global parameter applied to all vehicles.

v16  budgets x0.8               (CV-optimal lower end)
v17  budgets x1.15              (slightly larger budgets)
v18  fewer inspections: 1450    (only the riskiest vehicles)
v19  more inspections: 1950     (cast a wider net)
v20  homoscedastic budgets      (v12-style budgets on v14 flags: tests the sigma model)
v21  v9's flags + v14 budgets   (isolates the budget model vs v9)
v22  (dropped: 1685/1692 flags identical to v14, not worth a submission)
v23  budgets x0.9 and 1800 inspections (combined mild move)
"""
import numpy as np, pandas as pd
from policy import decide
from cv_eval import y, cost
from make_v12_v14 import em_adjust, write, q14, f14, mut, sig_t, z, eps, ALPHA

base_b = decide(q14, mut, z, ALPHA, 1.0, -1e9, sigma=sig_t)[1]   # hetero budget for every vehicle
homo_b = decide(q14, mut, eps, ALPHA, 1.0, -1e9)[1]


def flags_for(n):
    """Same policy as v14, alpha scaled so exactly ~n vehicles are flagged."""
    lo, hi = 0.05, 50.0
    for _ in range(40):
        mid = np.sqrt(lo * hi)
        k = decide(q14, mut, eps, ALPHA * mid, 1.0, 0.0)[0].sum()
        lo, hi = (mid, hi) if k < n else (lo, mid)
    return decide(q14, mut, eps, ALPHA * hi, 1.0, 0.0)[0]


def keep(f, b):
    return np.where(f == 1, b, 0.0)


write("v16_budget080", f14, keep(f14, base_b * 0.8))
write("v17_budget115", f14, keep(f14, base_b * 1.15))
f18 = flags_for(1450); write("v18_insp1450", f18, keep(f18, base_b))
f19 = flags_for(1950); write("v19_insp1950", f19, keep(f19, base_b))
write("v20_homo_budget", f14, keep(f14, homo_b))
f9 = pd.read_csv("submission_v9_risk6.csv").flag.to_numpy()
write("v21_v9flags_v14budget", f9, keep(f9, base_b))
score = q14 * np.exp(mut + 0.5 * np.var(eps))
f22 = np.zeros(len(q14), int); f22[np.argsort(-score)[:int(f14.sum())]] = 1
# v22 not written: nearly identical to v14
f23 = flags_for(1800); write("v23_budget090_insp1800", f23, keep(f23, base_b * 0.9))

for name, f in [("v18", f18), ("v19", f19), ("v21", f9), ("v22", f22), ("v23", f23)]:
    print(f"{name}: overlap with v14 flags {int((f & f14).sum())}/{int(f14.sum())}")

### `make_submission.py`
Build the final submission from saved ensemble predictions (run oof_models.py first).

In [ ]:
%%writefile make_submission.py
"""Build the final submission from saved ensemble predictions (run oof_models.py first)."""
import numpy as np, pandas as pd
from policy import decide

ALPHA, EPS_SCALE, MARGIN = 1.5, 1.0, 2.0   # chosen on out-of-fold cost (0.838 vs 3.49 do-nothing)
tr = pd.read_csv("data/train.csv", usecols=["failure", "repair_cost"])
te = pd.read_csv("data/test.csv", usecols=["id"])
y = tr.failure.to_numpy(); c = tr.repair_cost.fillna(0).to_numpy()
mu = np.load("oof/mu_train.npy"); mut = np.load("oof/mu_test.npy")
eps = np.log(c[y == 1]) - mu[y == 1]
pt = np.mean([np.load(f"oof/pt_{n}_w.npy") for n in ("lgb", "xgb", "cat")], 0)
f, b = decide(pt, mut, eps, ALPHA, EPS_SCALE, MARGIN)
sub = pd.DataFrame({"id": te.id, "flag": f.astype(int), "budget": np.round(b, 2)})
samp = pd.read_csv("data/sample_submission.csv")
assert list(sub.columns) == list(samp.columns) and (sub.id.values == samp.id.values).all()
assert sub.flag.isin([0, 1]).all() and np.isfinite(sub.budget).all() and (sub.budget >= 0).all()
assert (sub.loc[sub.flag == 0, "budget"] == 0).all()
sub.to_csv("submission_v5_ensemble.csv", index=False)
print("flagged", sub.flag.sum(), "of", len(sub), "| median budget", sub.loc[sub.flag == 1, "budget"].median())

### `make_v26_v28.py`
v26-v28: refinements of v9, the best leaderboard version (17,664).

In [ ]:
%%writefile make_v26_v28.py
"""v26-v28: refinements of v9, the best leaderboard version (17,664).

Leaderboard so far: v21 (v9 flags + new budget model) 17,677 ~ v9, and v14
(different ranking) 17,765, so v9's ranking is the one to build on.

v26  v9 policy (risk x6) on 15 full-data classifiers instead of 6 (more bagging,
     same settings; 99.97% correlated with v9's probabilities)
v27  v9 flags, budgets x0.7   } budget-level probe on the best flags. Flagged
v28  v9 flags, budgets x1.4   } vehicles are mostly healthy, and each pays 0.02*budget.

Rejected: ridge component in the cost model. CV RMSE improved 0.807 -> 0.790, but on
test it moves predictions 5x more than on train (mean |delta| 0.55 vs 0.11): linear
extrapolation into the test depot's heavier-usage range.
"""
import numpy as np, pandas as pd
from policy import decide
from cv_eval import y, cost

te = pd.read_csv("data/test.csv", usecols=["id"])
mu = np.load("oof/mu_train.npy"); eps = np.log(cost[y == 1]) - mu[y == 1]
mut = np.load("oof/full_mut.npy")


def write(name, f, b):
    sub = pd.DataFrame({"id": te.id, "flag": f.astype(int), "budget": np.round(np.where(f == 1, b, 0), 2)})
    assert len(sub) == 20000 and sub.flag.isin([0, 1]).all() and (sub.budget >= 0).all()
    sub.to_csv(f"submission_{name}.csv", index=False)
    print(f"{name}: inspections {int(f.sum())}, mean budget {sub.budget[sub.flag == 1].mean():.0f}")
    return f


f9, b9 = decide(np.load("oof/full_pt.npy"), mut, eps, 6.0, 1.0, 0.0)
f26 = write("v26_risk6_bag15", *decide(np.load("oof/full_pt5.npy"), mut, eps, 6.0, 1.0, 0.0))
print("v26 flags shared with v9:", int((f26 & f9).sum()), "of", int(f9.sum()))
write("v27_v9_budget070", f9, b9 * 0.7)
write("v28_v9_budget140", f9, b9 * 1.4)

### `make_v29.py`
v29: v9's policy (risk x6, same cost model and budgets rule) on the classifier ensemble retrained with histogram shape features (hist_features.py).

In [ ]:
%%writefile make_v29.py
"""v29: v9's policy (risk x6, same cost model and budgets rule) on the classifier ensemble
retrained with histogram shape features (hist_features.py).

CV (LightGBM + CatBoost, 5-fold x 2 seeds): OOF cost 0.8359 -> 0.8085 (-3.3%).
Leaderboard calibration after v27 says v9's risk x6 is the right setting for this
probability scale; the new ensemble has the same mean test probability (0.0211 vs 0.0212).
"""
import numpy as np, pandas as pd
from policy import decide
from cv_eval import y, cost

te = pd.read_csv("data/test.csv", usecols=["id"])
mu = np.load("oof/mu_train.npy"); eps = np.log(cost[y == 1]) - mu[y == 1]
mut = np.load("oof/full_mut.npy")
f, b = decide(np.load("oof/full_pt_shapes.npy"), mut, eps, 6.0, 1.0, 0.0)
sub = pd.DataFrame({"id": te.id, "flag": f.astype(int), "budget": np.round(b, 2)})
assert len(sub) == 20000 and sub.flag.isin([0, 1]).all() and (sub.budget >= 0).all()
assert (sub.loc[sub.flag == 0, "budget"] == 0).all()
sub.to_csv("submission_v29_shapes.csv", index=False)
f9 = pd.read_csv("submission_v9_risk6.csv").flag.to_numpy()
print(f"v29: inspections {f.sum()}, mean budget {b[f == 1].mean():.0f}, flags shared with v9 {(f & f9).sum()} of {f9.sum()}")

### `make_variants.py`
Write submission variants from the full-data fit (run full_fit.py first).

In [ ]:
%%writefile make_variants.py
"""Write submission variants from the full-data fit (run full_fit.py first)."""
import numpy as np, pandas as pd
from policy import decide

tr = pd.read_csv("data/train.csv", usecols=["failure", "repair_cost"])
te = pd.read_csv("data/test.csv", usecols=["id"])
y = tr.failure.to_numpy(); c = tr.repair_cost.fillna(0).to_numpy()
mu = np.load("oof/mu_train.npy"); eps = np.log(c[y == 1]) - mu[y == 1]
V = {
    # quantile-matched test sensors, same policy settings as v5
    "v6_qmatch": (np.load("oof/full_ptq.npy"), np.load("oof/full_mutq.npy"), 1.5, 1.0, 2.0),
    # v5 inputs but more inspections (use if v6 scores worse than v5)
    "v7_aggressive": (np.load("oof/full_pt.npy"), np.load("oof/full_mut.npy"), 2.5, 1.0, 0.0),
}
for name, (p, m, a, s, mg) in V.items():
    f, b = decide(p, m, eps, a, s, mg)
    sub = pd.DataFrame({"id": te.id, "flag": f.astype(int), "budget": np.round(b, 2)})
    assert sub.flag.isin([0, 1]).all() and (sub.budget >= 0).all() and len(sub) == 20000
    sub.to_csv(f"submission_{name}.csv", index=False)
    fl = sub.flag == 1
    print(f"{name}: flags={fl.sum()}  sum(10+0.02*budget)={10*fl.sum()+0.02*sub.budget[fl].sum():.0f}")

### `pseudo_test.py`
Pseudo-test validation: reweight training vehicles by the train->test density ratio (domain classifier odds) and score policies on the reweighted OOF predictions.

In [ ]:
%%writefile pseudo_test.py
"""Pseudo-test validation: reweight training vehicles by the train->test density ratio
(domain classifier odds) and score policies on the reweighted OOF predictions.

Result: reweighted failure rate is 1.9% (EM estimate for test: 3.5%), and the
reweighted cost prefers risk x1.5 while the leaderboard improved steadily up to x6.
The test depot's extra risk is mostly outside the training range, so covariate
reweighting cannot reproduce it; this proxy is not used to pick the risk level.
It does agree with CV that budgets x0.8-1.0 are best.
"""
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from fleet_prep import load
from policy import decide, competition_cost
from cv_eval import y, cost, weights

tr, te, X, Xt = load()
A = pd.concat([X, Xt]); d = np.r_[np.zeros(len(X)), np.ones(len(Xt))]
m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=100,
                       colsample_bytree=0.5, subsample=0.8, subsample_freq=1, verbosity=-1)
pd_ = cross_val_predict(m, A, d, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
pt = np.clip(pd_[:len(X)], 1e-4, 1 - 1e-4)
r = np.clip(pt / (1 - pt) * len(X) / len(Xt), 0, None); r = np.minimum(r, np.quantile(r, 0.999))
print("reweighted failure rate %.4f" % np.average(y, weights=weights * r))
mu = np.load("oof/mu_train.npy"); eps = np.log(cost[y == 1]) - mu[y == 1]
p = np.mean([np.load(f"oof/p_{n}_w.npy") for n in ("lgb", "xgb", "cat")], 0)
for a in [1.5, 2.5, 4, 6, 8]:
    cc = competition_cost(cost, *decide(p, mu, eps, a, 1.0, 0))
    print(f"alpha {a}: train {cc.mean():.4f}  pseudo-test {np.average(cc, weights=r):.4f}")

### `sigma_model.py`
Heteroscedastic repair-cost spread: predict |log-cost residual| per vehicle.

In [ ]:
%%writefile sigma_model.py
"""Heteroscedastic repair-cost spread: predict |log-cost residual| per vehicle.

Saves oof/sigma_train.npy (OOF for failed vehicles, fold-average elsewhere),
oof/sigma_test.npy and oof/z.npy (standardised residuals of failed vehicles).
"""
import numpy as np, lightgbm as lgb
from sklearn.model_selection import KFold
from fleet_prep import load
from cv_eval import y, cost

_, _, X, Xt = load()
pos = np.where(y == 1)[0]
mu = np.load("oof/mu_train.npy")
res = np.log(cost[pos]) - mu[pos]
Xp = X.iloc[pos]
oof = np.zeros(len(pos)); s_all = np.zeros(len(X)); s_test = np.zeros(len(Xt))
SEEDS, NF = [0, 1, 2], 5
for seed in SEEDS:
    for a, b in KFold(NF, shuffle=True, random_state=seed).split(Xp):
        m = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.03, num_leaves=4, min_child_samples=15,
                              random_state=seed, verbosity=-1).fit(Xp.iloc[a], np.abs(res[a]))
        oof[b] += m.predict(Xp.iloc[b]) / len(SEEDS)
        s_all += m.predict(X) / (len(SEEDS) * NF)
        s_test += m.predict(Xt) / (len(SEEDS) * NF)
s_all[pos] = oof
s_all, s_test = np.maximum(s_all, 0.05), np.maximum(s_test, 0.05)
np.save("oof/sigma_train.npy", s_all); np.save("oof/sigma_test.npy", s_test)
np.save("oof/z.npy", res / s_all[pos])
print("sigma train mean %.3f  test mean %.3f" % (s_all.mean(), s_test.mean()))

### `sigma_shapes.py`
Per-vehicle spread of the log repair cost, with shape features, on top of the v30 cost model. Scored by OOF competition cost (v29/v30 classifier). Saves oof/sig2_train.npy, oof/sig2_test.npy, oof/z2.npy.

In [ ]:
%%writefile sigma_shapes.py
"""Per-vehicle spread of the log repair cost, with shape features, on top of the v30 cost
model. Scored by OOF competition cost (v29/v30 classifier). Saves oof/sig2_train.npy,
oof/sig2_test.npy, oof/z2.npy."""
import numpy as np, lightgbm as lgb
from sklearn.model_selection import KFold
from hist_features import with_shapes
from cv_eval import y, cost
from policy import decide, competition_cost

_, _, X, Xt = with_shapes()
pos = np.where(y == 1)[0]
mu = np.load("oof/mu_train_shapes.npy"); res = np.log(cost[pos]) - mu[pos]
P = (np.load("oof/lgb2_shapes.npy") + np.load("oof/cat2_shapes.npy")) / 2
Xp = X.iloc[pos]
for leaves, n in [(4, 200), (3, 150)]:
    oof = np.zeros(len(pos)); s_all = np.zeros(len(X)); s_te = np.zeros(len(Xt))
    for seed in [0, 1, 2]:
        for a, b in KFold(5, shuffle=True, random_state=seed).split(Xp):
            m = lgb.LGBMRegressor(n_estimators=n, learning_rate=0.03, num_leaves=leaves, min_child_samples=15,
                                  random_state=seed, verbosity=-1).fit(Xp.iloc[a], np.abs(res[a]))
            oof[b] += m.predict(Xp.iloc[b]) / 3; s_all += m.predict(X) / 15; s_te += m.predict(Xt) / 15
    s_all[pos] = oof; s_all = np.maximum(s_all, 0.05); s_te = np.maximum(s_te, 0.05)
    z = res / s_all[pos]
    for al in [1, 1.25, 1.5]:
        c = competition_cost(cost, *decide(P, mu, z, al, 1.0, 0.0, sigma=s_all)).sum()
        print(f"leaves {leaves}: alpha {al} OOF cost {c:.0f}", flush=True)
    if leaves == 4:
        np.save("oof/sig2_train.npy", s_all); np.save("oof/sig2_test.npy", s_te); np.save("oof/z2.npy", z)
print("homoscedastic reference:", min(competition_cost(cost, *decide(P, mu, res, al, 1.0, 0.0)).sum() for al in [1, 1.25, 1.5]))

### `smooth_clf.py`
Smooth failure model: spline logistic regression on bounded histogram-shape features plus log operating total (same inputs as the smooth cost model behind v33/v34). Scored by OOF competition cost with the v34 budget model, alone and blended with the tree ensemble.

In [ ]:
%%writefile smooth_clf.py
"""Smooth failure model: spline logistic regression on bounded histogram-shape features plus
log operating total (same inputs as the smooth cost model behind v33/v34). Scored by
OOF competition cost with the v34 budget model, alone and blended with the tree ensemble."""
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.metrics import roc_auc_score, average_precision_score
from cost_linear_shapes import F, Ft
from cv_eval import y, cost, weights
from policy import decide, competition_cost

pos = y == 1; lc = np.log(cost[pos])
tree = np.load("oof/mu_train_shapes.npy")
mu_b = np.load("oof/bb_k3d1_w1.0_train.npy")
PT = (np.load("oof/lgb2_shapes.npy") + np.load("oof/cat2_shapes.npy")) / 2


def score(p):
    best = None
    for al in [0.75, 1, 1.25, 1.5, 2]:
        f, _ = decide(p, tree, lc - tree[pos], al, 1.0, 0.0)
        _, b = decide(p, mu_b, lc - mu_b[pos], al, 1.0, -1e9)
        c = competition_cost(cost, f, np.where(f == 1, b, 0)).sum()
        if best is None or c < best[0]: best = (c, al)
    return best


def make(C, knots):
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         SplineTransformer(n_knots=knots, degree=1), StandardScaler(),
                         LogisticRegression(C=C, max_iter=3000))


if __name__ == "__main__":
    print("tree ensemble (lgb+cat shapes): AUC %.4f AP %.4f cost %s" % (roc_auc_score(y, PT), average_precision_score(y, PT), score(PT)))
    for C, knots in [(0.01, 3), (0.003, 3), (0.01, 5)]:
        oof = np.zeros(len(y)); tst = np.zeros(len(Ft))
        for a, b in StratifiedKFold(5, shuffle=True, random_state=0).split(F, y):
            m = make(C, knots).fit(F.iloc[a], y[a], logisticregression__sample_weight=weights[a])
            oof[b] = m.predict_proba(F.iloc[b])[:, 1]; tst += m.predict_proba(Ft)[:, 1] / 5
        print(f"smooth C{C} knots{knots}: AUC {roc_auc_score(y, oof):.4f} AP {average_precision_score(y, oof):.4f} cost {score(oof)}", flush=True)
        for w in [0.2, 0.35, 0.5]:
            bl = (1 - w) * PT + w * oof
            print(f"    blend w{w}: AUC {roc_auc_score(y, bl):.4f} AP {average_precision_score(y, bl):.4f} cost {score(bl)}", flush=True)
        np.save(f"oof/sclf_C{C}_k{knots}_train.npy", oof); np.save(f"oof/sclf_C{C}_k{knots}_test.npy", tst)

### `tune.py`
Tune the decision policy on out-of-fold predictions and compare to baselines.

In [ ]:
%%writefile tune.py
"""Tune the decision policy on out-of-fold predictions and compare to baselines."""
import itertools, sys
import numpy as np
import pandas as pd
from policy import decide, competition_cost

tr = pd.read_csv("data/train.csv", usecols=["failure", "repair_cost", "followup_days"])
y = tr.failure.to_numpy(); c = tr.repair_cost.fillna(0).to_numpy()
pos = y == 1
mu = np.load("oof/mu_train.npy")
eps = np.log(c[pos]) - mu[pos]


def P(tag, names=("lgb", "xgb", "cat")):
    return np.mean([np.load(f"oof/p_{n}_{tag}.npy") for n in names], 0)


def score(flag, b):
    return competition_cost(c, flag, b).mean()


print("Do nothing (never inspect):          %.4f" % c.mean())
q90 = np.quantile(c[pos], 0.9)
for tag in ["w", "nw"]:
    for names in [("lgb",), ("xgb",), ("cat",), ("lgb", "xgb", "cat")]:
        p = P(tag, names)
        # notebook-style baseline: threshold on p*exp(mu), fixed q90 budget
        e = p * np.exp(mu)
        best = min(score((e > t).astype(int), np.where(e > t, q90, 0))
                   for t in np.quantile(e, np.linspace(0.9, 0.9995, 200)))
        f, b = decide(p, mu, eps)
        print(f"{tag:>2} {'+'.join(names):12s} threshold+q90 best={best:.4f} | "
              f"optimal policy (no tuning)={score(f, b):.4f} flags={f.sum()}")

tag = sys.argv[1] if len(sys.argv) > 1 else "w"
p = P(tag)
rows = []
for a, s, m, bm in itertools.product([0.6, 0.8, 1.0, 1.2, 1.5], [0.8, 1.0, 1.2],
                                     [-2, 0, 2, 5], [0.8, 1.0, 1.2]):
    f, b = decide(p, mu, eps, a, s, m, bm)
    rows.append((a, s, m, bm, score(f, b), f.sum()))
r = pd.DataFrame(rows, columns=["alpha", "eps_scale", "margin", "budget_mult", "cost", "flags"])
print(r.sort_values("cost").head(10).to_string(index=False))

### `tune_budget.py`
Compare budget models on failed vehicles via CV pinball loss at tau=0.909 (the competition's under/over penalty ratio 0.20/0.22).

In [ ]:
%%writefile tune_budget.py
"""Compare budget models on failed vehicles via CV pinball loss at tau=0.909
(the competition's under/over penalty ratio 0.20/0.22)."""
import numpy as np, lightgbm as lgb
from sklearn.model_selection import KFold
from fleet_prep import load
from cv_eval import y, cost
TAU = 0.2 / 0.22
_, _, X, _ = load()
pos = np.where(y == 1)[0]; Xp = X.iloc[pos]; C = cost[pos]; lc = np.log(C)
mu = np.load("oof/mu_train.npy")[pos]
def pin(b): return np.mean(0.2 * np.maximum(C - b, 0) + 0.02 * np.maximum(b - C, 0))
print("global quantile budget           %.2f" % pin(np.quantile(C, TAU)))
# current: exp(mu + quantile of OOF residuals)
res = lc - mu
print("current mu + residual quantile   %.2f" % pin(np.exp(mu + np.quantile(res, TAU))))
for s in [0.8, 1.2]:
    print(f"  residual scale {s}              %.2f" % pin(np.exp(mu + s * np.quantile(res, TAU))))
# direct quantile regression on log cost, with OOF
for leaves, n in [(4, 200), (7, 300), (7, 600)]:
    oof = np.zeros(len(pos))
    for seed in [0, 1, 2]:
        for a, b in KFold(5, shuffle=True, random_state=seed).split(Xp):
            m = lgb.LGBMRegressor(objective="quantile", alpha=TAU, n_estimators=n, learning_rate=0.03,
                                  num_leaves=leaves, min_child_samples=10, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=0.5, random_state=seed, verbosity=-1)
            m.fit(Xp.iloc[a], lc[a]); oof[b] += m.predict(Xp.iloc[b]) / 3
    print(f"LGBM quantile on log cost ({leaves} leaves, {n} trees) %.2f" % pin(np.exp(oof)))
# heteroscedastic: model |residual| and scale
oof_s = np.zeros(len(pos))
for seed in [0, 1, 2]:
    for a, b in KFold(5, shuffle=True, random_state=seed).split(Xp):
        m = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.03, num_leaves=4, min_child_samples=15,
                              random_state=seed, verbosity=-1).fit(Xp.iloc[a], np.abs(res[a]))
        oof_s[b] += m.predict(Xp.iloc[b]) / 3
z = res / np.maximum(oof_s, 0.05)
print("heteroscedastic mu + s(x)*q(z)    %.2f" % pin(np.exp(mu + np.maximum(oof_s, 0.05) * np.quantile(z, TAU))))

### `tune_lgb.py`
Small CV search over LightGBM settings (same folds for every config).

In [ ]:
%%writefile tune_lgb.py
"""Small CV search over LightGBM settings (same folds for every config)."""
import numpy as np, lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from fleet_prep import load
from cv_eval import y, cost, weights, report

_, _, X, _ = load()
mu = np.load("oof/mu_train.npy"); eps = np.log(cost[y == 1]) - mu[y == 1]
base = dict(n_estimators=600, learning_rate=0.02, num_leaves=15, min_child_samples=40, subsample=0.8,
            subsample_freq=1, colsample_bytree=0.5, reg_lambda=5)
grid = {
    "base": {},
    "leaves7": dict(num_leaves=7),
    "leaves31": dict(num_leaves=31, min_child_samples=20),
    "trees1200_lr01": dict(n_estimators=1200, learning_rate=0.01),
    "col0.3": dict(colsample_bytree=0.3),
    "minchild80": dict(min_child_samples=80),
    "lambda20": dict(reg_lambda=20),
    "trees300": dict(n_estimators=300),
    "trees1000": dict(n_estimators=1000),
    "pos_w3": dict(scale_pos_weight=3),
}
folds = list(StratifiedKFold(5, shuffle=True, random_state=0).split(X, y))
for name, kw in grid.items():
    oof = np.zeros(len(X))
    for a, b in folds:
        m = lgb.LGBMClassifier(**{**base, **kw}, random_state=0, verbosity=-1)
        m.fit(X.iloc[a], y[a], sample_weight=weights[a])
        oof[b] = m.predict_proba(X.iloc[b])[:, 1]
    report(name, oof, mu, eps)